# Music Annotation Project V9.0A
## Duplicate-Aware Mask-Authority Transfer Audit

This revision fixes the Notebook 06/07 audit problem by grouping all records with the same `source_md5`, inventorying duplicates, and resolving only an unambiguous Notebook 06 → Notebook 07 lineage. It never silently keeps the first or last duplicate.

Canonical identity:
- Notebook 06 files keep the original source stem.
- Notebook 07 supplies the downstream `PageXXX` identity.
- Cross-stage linkage is `source_md5`.

The experiment remains development/shadow evidence only. It does not authorise automatic mask authority.

In [ ]:
# PROVENANCE: standard project Colab bootstrap.
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib

## 1. Configuration

In [ ]:
# PROVENANCE: paths from Notebooks 06, 07 and 09; V9.0A run controls.
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, re, sys, platform
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
DRAFT_AUDIT_DIR=ROOT/'human_in_the_middle'/'draft_audit'
REVIEW_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'
CLEAN_DIR=ROOT/'source'/'clean'
OUTPUT_ROOT=ROOT/'mask_authority_transfer_v9_0a'
PAGE_ALLOWLIST=None
# Established prior mask-authority gate. V8.2b records 0.90 as the exploratory automatic candidate.
ESTABLISHED_YOLO_GATE=0.90
ESTABLISHED_YOLO_GATE_POLICY_ID='V8.2b_AUTO_GE_090_RUN2'
MASK_DILATION=2; CROP_MARGIN=48; GEOMETRY_THRESHOLD=80; BAND_WIDTH=4
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v9_0a_duplicate_aware'
OUT=OUTPUT_ROOT/'runs'/RUN_ID
INPUTS=OUT/'01_inputs'; MASKS=OUT/'02_mask_arms'; OUTPUTS=OUT/'03_corrected_pages'; LOCKS=OUT/'04_locks'; SCORE=OUT/'05_scoring'; VISUAL=OUT/'06_visual_audit'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MASKS,OUTPUTS,LOCKS,SCORE,VISUAL,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
ARMS=['reviewed_comparator','raw_yolo','confidence_gated_yolo','reviewer_corrected_yolo']
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
print('Output:',OUT)

## 2. Integrity helpers

In [ ]:
# PROVENANCE: shared project integrity helpers.
def require(x,msg):
    if not bool(x):
        try:(AUDIT/'fatal_failure.json').write_text(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'message':str(msg)},indent=2))
        except Exception:pass
        raise RuntimeError(msg)
def md5_file(p):
    h=hashlib.md5()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def sha256_file(p):
    p=Path(p);require(p.exists(),f'Missing file: {p}');h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(p,v):
    p=Path(p);t=Path(str(p)+'.tmp');t.write_text(json.dumps(v,indent=2,sort_keys=True,default=str));t.replace(p)
def read_gray(p):
    a=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE);require(a is not None,f'Unreadable image: {p}');return a
def binary(a):return np.where(a>0,255,0).astype(np.uint8)

## 3. Load, group and diagnose all Notebook 06/07 audits

In [ ]:
# PROVENANCE: adapted from Notebook 09 audit loading. Duplicates are grouped and reported, not rejected during dictionary construction.
def load_audits(folder,label):
    rows=[];bad=[]
    for p in sorted(Path(folder).glob('*.json')):
        try:
            r=json.loads(p.read_text());require(isinstance(r,dict),'audit root must be object');r['_audit_path']=str(p);r['_audit_mtime_ns']=p.stat().st_mtime_ns;r['_audit_type']=label;rows.append(r)
        except Exception as e:bad.append({'audit_type':label,'audit_path':str(p),'error':str(e)})
    return rows,bad
def group_by_source_md5(records,label):
    grouped={}
    for r in records:
        h=str(r.get('source_md5','')).strip().lower();require(h,f'{label} audit lacks source_md5: {r["_audit_path"]}')
        grouped.setdefault(h,[]).append(r)
    return grouped
def path_value(r,key):
    raw=r.get(key);return Path(raw) if raw else None
def record_signature(r,kind):
    keys=['source_file','draft_mask','confidence_map','model_path','model_md5','prediction_confidence'] if kind=='draft' else ['page','source','aligned_output','corrected_mask_output']
    return object_hash({k:r.get(k) for k in keys})
def valid_record(r,kind):
    needed=['source_file','draft_mask','confidence_map'] if kind=='draft' else ['aligned_output','corrected_mask_output']
    missing=[k for k in needed if not r.get(k) or not Path(r[k]).exists()]
    return len(missing)==0,missing

drafts,dbad=load_audits(DRAFT_AUDIT_DIR,'draft');reviews,rbad=load_audits(REVIEW_AUDIT_DIR,'review')
if dbad+rbad:
    pd.DataFrame(dbad+rbad).to_csv(AUDIT/'audit_parse_failures.csv',index=False);raise RuntimeError('Unreadable HITL audit records found.')
draft_groups=group_by_source_md5(drafts,'Notebook 06 draft');review_groups=group_by_source_md5(reviews,'Notebook 07 review')
paired=sorted(set(draft_groups)&set(review_groups));require(paired,'No paired source_md5 values found.')
inv=[]
for h in paired:
    for kind,records in [('draft',draft_groups[h]),('review',review_groups[h])]:
        for r in records:
            ok,missing=valid_record(r,kind)
            inv.append({'source_md5':h,'audit_type':kind,'audit_path':r['_audit_path'],'mtime_ns':r['_audit_mtime_ns'],'signature':record_signature(r,kind),'valid':ok,'missing_fields_or_paths':'|'.join(missing),'page':r.get('page'),'source':r.get('source_file',r.get('source')),'draft_mask':r.get('draft_mask'),'confidence_map':r.get('confidence_map'),'aligned_output':r.get('aligned_output'),'corrected_mask_output':r.get('corrected_mask_output')})
inv=pd.DataFrame(inv);inv.to_csv(AUDIT/'all_paired_audit_candidates.csv',index=False)
dup=inv.groupby(['source_md5','audit_type']).filter(lambda g:len(g)>1)
dup.to_csv(AUDIT/'duplicate_source_md5_audits.csv',index=False)
print('Paired hashes:',len(paired),'duplicate candidate rows:',len(dup))

## 4. Resolve one authoritative record per stage without first/last guessing

In [ ]:
# PROVENANCE: NEW V9.0A duplicate resolution. Identical replicas may collapse; conflicting valid records stop.
def resolve_group(records,kind,h):
    valid=[];invalid=[]
    for r in records:
        ok,missing=valid_record(r,kind)
        (valid if ok else invalid).append((r,missing))
    require(valid,f'{h}: no valid {kind} audit; invalid={[(r["_audit_path"],m) for r,m in invalid]}')
    by_sig={}
    for r,_ in valid:by_sig.setdefault(record_signature(r,kind),[]).append(r)
    if len(by_sig)==1:
        replicas=next(iter(by_sig.values()))
        # Deterministic choice only among semantically identical records.
        chosen=sorted(replicas,key=lambda r:r['_audit_path'])[0]
        status='unique' if len(replicas)==1 else 'identical_replicas_collapsed'
        return chosen,status,len(replicas)
    raise RuntimeError(f'{h}: conflicting valid {kind} audits. Inspect {AUDIT/"duplicate_source_md5_audits.csv"}; no record was selected.')

resolved=[];decisions=[]
for h in paired:
    d,ds,dn=resolve_group(draft_groups[h],'draft',h);r,rs,rn=resolve_group(review_groups[h],'review',h)
    page=str(r.get('page','')).strip() or Path(r['_audit_path']).stem
    require(re.fullmatch(r'Page\d+',page,re.I),f'{h}: invalid Notebook 07 page identity {page!r}')
    sp=Path(d['source_file']);rp=Path(d['draft_mask']);cp=Path(d['confidence_map']);op=Path(r['aligned_output']);mp=Path(r['corrected_mask_output'])
    require(md5_file(sp).lower()==h,f'{page}: Notebook 06 source hash mismatch')
    if r.get('source') and Path(r['source']).exists():require(md5_file(Path(r['source'])).lower()==h,f'{page}: Notebook 07 source hash mismatch')
    obs=read_gray(op);raw=read_gray(rp);reviewed=read_gray(mp);conf=np.load(cp).astype(np.float32)
    require(obs.shape==raw.shape==reviewed.shape==conf.shape,f'{page}: shape mismatch')
    require(np.isfinite(conf).all(),f'{page}: non-finite confidence')
    resolved.append({'page':page,'source_md5':h,'original_source_path':str(sp),'observed_path':str(op),'raw_yolo_mask_path':str(rp),'confidence_map_path':str(cp),'reviewed_mask_path':str(mp),'draft_audit_path':d['_audit_path'],'review_audit_path':r['_audit_path'],'draft_resolution':ds,'review_resolution':rs,'draft_replica_count':dn,'review_replica_count':rn,'draft_prediction_confidence':d.get('prediction_confidence'),'draft_model_path':d.get('model_path'),'draft_model_md5':d.get('model_md5')})
    decisions.append({'source_md5':h,'page':page,'draft_status':ds,'draft_chosen':d['_audit_path'],'review_status':rs,'review_chosen':r['_audit_path']})
resolution=pd.DataFrame(resolved).sort_values('page').reset_index(drop=True)
require(not resolution.page.duplicated().any(),'Duplicate PageXXX identities remain after source-hash resolution.')
if PAGE_ALLOWLIST is not None:
    missing=sorted(set(PAGE_ALLOWLIST)-set(resolution.page));require(not missing,f'Allowlisted pages missing: {missing}');resolution=resolution[resolution.page.isin(PAGE_ALLOWLIST)].copy()
require(len(resolution)>0,'No pages remain.')
resolution.to_csv(AUDIT/'hitl_input_resolution.csv',index=False);pd.DataFrame(decisions).to_csv(AUDIT/'audit_resolution_decisions.csv',index=False)
display(resolution[['page','source_md5','draft_resolution','review_resolution','raw_yolo_mask_path','confidence_map_path','reviewed_mask_path']])

## 5. Validate established gate and canonical correction functions

In [ ]:
# PROVENANCE: V8.2b establishes AUTO_THRESHOLD=0.900 and MIN_PRINT_RUN=2 as its exploratory automatic candidate; V9.0A inherits the threshold as a declared experiment input.
require(0.5<=float(ESTABLISHED_YOLO_GATE)<=1.0,'Invalid established gate')
require(str(ESTABLISHED_YOLO_GATE_POLICY_ID).strip(),'Missing gate policy ID')
rawvals=pd.to_numeric(resolution.draft_prediction_confidence,errors='coerce').dropna().unique().tolist()
require(len(rawvals)<=1,f'Notebook 06 raw thresholds disagree: {rawvals}')
raw_threshold=float(rawvals[0]) if rawvals else None
require(raw_threshold is None or ESTABLISHED_YOLO_GATE>raw_threshold,'Gated threshold must exceed raw draft threshold.')
atomic_json(AUDIT/'yolo_gate_identity.json',{'gate':ESTABLISHED_YOLO_GATE,'policy_id':ESTABLISHED_YOLO_GATE_POLICY_ID,'not_optimised_in_v9_0a':True,'raw_draft_threshold':raw_threshold})

def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0];require(np.array_equal(out[mask==0],original[mask==0]),'outside-mask drift');return out
def prepare_mask(seed,dilation=MASK_DILATION):
    m=binary(seed)
    if dilation>0:m=cv2.dilate(m,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return m
def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False;merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop();rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else:rest.append((bx0,by0,bx1,by1))
            boxes=rest;merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes
def telea_reconstruct(obs,mask):
    result=obs.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=obs[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
    return immutable(obs,result,mask)
def background_surface(obs,mask,sigma=45.0):
    v=(mask==0)&(obs>=200);num=cv2.GaussianBlur(obs.astype(np.float32)*v,(0,0),sigma);den=cv2.GaussianBlur(v.astype(np.float32),(0,0),sigma);fallback=cv2.GaussianBlur(obs,(0,0),sigma);return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
def support_maps(obs,mask):
    dark=(obs<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0;n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);cc=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:cc[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8);return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(cc.astype(np.uint8),k)>0
def geometry_t80(obs,mask,telea):
    paper=background_surface(obs,mask);norm=cv2.divide(obs,np.maximum(paper,1),scale=255);h,v,c=support_maps(obs,mask);m=mask>0;pres=m&(h|v|c)&(obs<200);rem=m&~pres&(obs>=GEOMETRY_THRESHOLD)&(norm>=GEOMETRY_THRESHOLD);amb=m&~pres&~rem;result=obs.copy();result[rem]=paper[rem];result[amb]=telea[amb];result[pres]=obs[pres];return immutable(obs,result,mask),{'preserve':pres,'remove':rem,'ambiguous':amb}

## 6. Execute arms and lock outputs

In [ ]:
# PROVENANCE: V9.0A paired orchestration using canonical dilation-2, Telea and geometry_t80 functions.
runtime={};mr=[];orows=[];ed=[]
for _,r in resolution.iterrows():
    p=r.page;obs=read_gray(r.observed_path);raw=binary(read_gray(r.raw_yolo_mask_path));rev=binary(read_gray(r.reviewed_mask_path));conf=np.load(r.confidence_map_path).astype(np.float32);gated=np.where((raw>0)&(conf>=ESTABLISHED_YOLO_GATE),255,0).astype(np.uint8)
    arms={'reviewed_comparator':rev,'raw_yolo':raw,'confidence_gated_yolo':gated,'reviewer_corrected_yolo':rev.copy()};runtime[p]={}
    ed.append({'page':p,'reviewer_additions':int(((rev>0)&(raw==0)).sum()),'reviewer_erasures':int(((rev==0)&(raw>0)).sum()),'xor':int(((rev>0)^(raw>0)).sum())})
    for arm,seed in arms.items():
        mask=prepare_mask(seed);telea=telea_reconstruct(obs,mask);corr,routes=geometry_t80(obs,mask,telea);d=OUTPUTS/arm/p;d.mkdir(parents=True,exist_ok=False);md=MASKS/arm;md.mkdir(parents=True,exist_ok=True)
        cv2.imwrite(str(md/f'{p}.png'),seed)
        for n,a in [('observed',obs),('seed',seed),('prepared_mask',mask),('corrected',corr),('preserve',routes['preserve'].astype(np.uint8)*255),('remove',routes['remove'].astype(np.uint8)*255),('ambiguous',routes['ambiguous'].astype(np.uint8)*255)]:cv2.imwrite(str(d/f'{n}.png'),a)
        require(np.array_equal(corr[mask==0],obs[mask==0]),f'{p}/{arm}: drift')
        runtime[p][arm]={'seed':seed,'mask':mask,'corrected':corr}
        mr.append({'page':p,'arm':arm,'seed_pixels':int((seed>0).sum()),'prepared_pixels':int((mask>0).sum()),'seed_hash':array_hash(seed),'mask_hash':array_hash(mask)})
        orows.append({'page':p,'arm':arm,'corrected_hash':array_hash(corr),'outside_drift':0,'changed_pixels':int((corr!=obs).sum())})
pd.DataFrame(mr).to_csv(LOCKS/'mask_arm_manifest.csv',index=False);pd.DataFrame(orows).to_csv(LOCKS/'corrected_output_manifest.csv',index=False);pd.DataFrame(ed).to_csv(SCORE/'reviewer_edits.csv',index=False)
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'pages':sorted(runtime),'hitl_resolution_sha256':sha256_file(AUDIT/'hitl_input_resolution.csv'),'duplicate_inventory_sha256':sha256_file(AUDIT/'duplicate_source_md5_audits.csv'),'mask_manifest_sha256':sha256_file(LOCKS/'mask_arm_manifest.csv'),'output_manifest_sha256':sha256_file(LOCKS/'corrected_output_manifest.csv'),'truth_loaded':False};lock['lock_id']=object_hash(lock);atomic_json(LOCKS/'prediction_and_output_lock.json',lock);print('Lock:',lock['lock_id'])

## 7. Post-lock comparison, optional truth, visuals and verdict

In [ ]:
# PROVENANCE: NEW V9.0A analysis after output lock.
def mm(c,r):
    c=c>0;r=r>0;i=int((c&r).sum());u=int((c|r).sum());return {'iou':i/max(u,1),'fp':int((c&~r).sum()),'fn':int((~c&r).sum()),'xor':int((c^r).sum())}
rows=[]
for p,a in runtime.items():
    rm=a['reviewed_comparator']['seed'];ro=a['reviewed_comparator']['corrected']
    for arm,z in a.items():
        q=mm(z['seed'],rm);q.update({'page':p,'arm':arm,'corrected_xor':int((z['corrected']!=ro).sum()),'corrected_mae':float(np.abs(z['corrected'].astype(np.int16)-ro.astype(np.int16)).mean())});rows.append(q)
paired_df=pd.DataFrame(rows);paired_df.to_csv(SCORE/'paired_evidence.csv',index=False);paired_df.groupby('arm',as_index=False).agg(pages=('page','nunique'),mean_iou=('iou','mean'),fp=('fp','sum'),fn=('fn','sum'),mask_xor=('xor','sum'),corrected_xor=('corrected_xor','sum')).to_csv(SCORE/'arm_summary.csv',index=False)
clean={p.stem:p for p in CLEAN_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if CLEAN_DIR.exists() else {};tr=[]
for p,a in runtime.items():
    if p not in clean:continue
    truth=read_gray(clean[p])
    for arm,z in a.items():
        diff=np.abs(truth.astype(np.float32)-z['corrected'].astype(np.float32));tr.append({'page':p,'arm':arm,'whole_mae':float(diff.mean()),'inside_mask_mae':float(diff[z['mask']>0].mean()) if (z['mask']>0).any() else np.nan})
if tr:pd.DataFrame(tr).to_csv(SCORE/'postlock_clean_scores.csv',index=False)
q=paired_df[paired_df.arm.isin(['raw_yolo','confidence_gated_yolo'])].copy();q['priority']=q.corrected_xor+5*q.fp+3*q.fn;q.sort_values('priority',ascending=False).to_csv(SCORE/'visual_queue.csv',index=False)
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0A-duplicate-aware','verdict':'PASS_EVIDENCE_READY_FOR_REVIEW','lock_id':lock['lock_id'],'automatic_mask_authority_authorised':False,'duplicate_resolution':'only unique or semantically identical replica groups accepted; conflicting valid groups stop'};atomic_json(OUT/'v9_0a_verdict.json',verdict);print(json.dumps(verdict,indent=2))

## 8. Final Upload Package

In [ ]:
# PROVENANCE: standard Final Upload Package.
required=[OUT/'v9_0a_verdict.json',AUDIT/'all_paired_audit_candidates.csv',AUDIT/'duplicate_source_md5_audits.csv',AUDIT/'audit_resolution_decisions.csv',AUDIT/'hitl_input_resolution.csv',AUDIT/'yolo_gate_identity.json',LOCKS/'mask_arm_manifest.csv',LOCKS/'corrected_output_manifest.csv',LOCKS/'prediction_and_output_lock.json',SCORE/'reviewer_edits.csv',SCORE/'paired_evidence.csv',SCORE/'arm_summary.csv',SCORE/'visual_queue.csv']
if (SCORE/'postlock_clean_scores.csv').exists():required.append(SCORE/'postlock_clean_scores.csv')
for p in required:require(p.exists(),f'Missing upload artefact: {p}')
(UPLOAD/'README.txt').write_text('V9.0A duplicate-aware mask-authority audit. Notebook 06/07 joined by source_md5. Duplicate semantic replicas collapse; conflicting valid records stop. Research evidence only.\n')
rows=[]
for src in required:
    dst=UPLOAD/src.name;shutil.copy2(src,dst);rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['file'] for x in rows))
zp=UPLOAD/'58_V9_0A_upload_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():
        if p.is_file() and p!=zp:z.write(p,p.name)
require(zp.exists() and zp.stat().st_size>0,'Bundle failed');print('Bundle:',zp)